In [18]:
# importações
import pandas as pd
import numpy as np

In [19]:
# Carregando base de dados
df = pd.read_parquet('../data/raw/ames_housing.parquet')

In [20]:
# Ajustando colunas temporais
COLS_TEMPORAIS = ['ano_construcao', 'ano_reforma', 'ano_garagem', 'mes_venda', 'ano_venda']

# Escalas das colunas ordinais (do menor para o maior nível)
ESCALA_QUALIDADE = ['Ruim', 'Regular', 'Médio', 'Bom', 'Excelente']
ESCALA_ACABAMENTO_PORAO = [
    'Sem acabamento', 'Baixa qualidade', 'Sala de recreação',
    'Habitável abaixo da média', 'Habitável média', 'Habitável boa',
]

ESCALAS_ORDINAIS = {
    'qualidade_geral': None,        # numérica, 1 a 10
    'conservacao_geral': None,      # numérica, 1 a 10
    'qualidade_externa': ESCALA_QUALIDADE,
    'conservacao_externa': ESCALA_QUALIDADE,
    'qualidade_porao': ESCALA_QUALIDADE,
    'conservacao_porao': ESCALA_QUALIDADE,
    'qualidade_aquecimento': ESCALA_QUALIDADE,
    'qualidade_cozinha': ESCALA_QUALIDADE,
    'qualidade_lareira': ESCALA_QUALIDADE,
    'qualidade_garagem': ESCALA_QUALIDADE,
    'conservacao_garagem': ESCALA_QUALIDADE,
    'qualidade_piscina': ESCALA_QUALIDADE,
    'exposicao_porao': ['Sem exposição', 'Mínima', 'Média', 'Boa'],
    'acabamento_porao_1': ESCALA_ACABAMENTO_PORAO,
    'acabamento_porao_2': ESCALA_ACABAMENTO_PORAO,
    'acabamento_garagem': ['Sem acabamento', 'Acabamento parcial', 'Acabada'],
    'acesso_pavimentado': ['Terra/cascalho', 'Parcialmente pavimentado', 'Pavimentado'],
    'formato_lote': ['Irregular', 'Moderadamente irregular', 'Levemente irregular', 'Regular'],
    'declividade': ['Acentuada', 'Moderada', 'Suave'],
    'infraestrutura': ['Só eletricidade', 'Eletricidade e gás', 'Sem rede de esgoto', 'Todos os serviços'],
    'sistema_eletrico': ['Misto', 'Fusíveis (ruim)', 'Fusíveis (regular)', 'Fusíveis (médio)', 'Disjuntores'],
    'funcionalidade': [
        'Apenas aproveitamento', 'Danos graves', 'Grandes deduções 2', 'Grandes deduções 1',
        'Deduções moderadas', 'Pequenas deduções 2', 'Pequenas deduções 1', 'Típica',
    ],
}
COLS_ORDINAIS = list(ESCALAS_ORDINAIS)

# lista de dicionários para armazenar informações sobre as colunas
col_info = []

# percorrendo as colunas do dataframe
for col in df.columns:
    serie = df[col]

    # tipo e valores (menor/maior nível, ou primeiro/último para nominais)
    if col in COLS_TEMPORAIS or pd.api.types.is_datetime64_any_dtype(serie):
        col_type = 'Temporal'
        values = [serie.min(), serie.max()]
    elif col in COLS_ORDINAIS:
        col_type = 'Ordinal'
        escala = ESCALAS_ORDINAIS[col]
        if escala is None:
            values = [serie.min(), serie.max()]
        else:
            presentes = [nivel for nivel in escala if nivel in set(serie.dropna())]
            values = [presentes[0], presentes[-1]] if presentes else [None, None]
    elif serie.dtype == 'int64':
        col_type = 'Discreta'
        values = [serie.min(), serie.max()]
    elif serie.dtype == 'float64':
        col_type = 'Contínua'
        values = [serie.min(), serie.max()]
    else:  # object, str, category, bool
        col_type = 'Nominal'
        uniques = serie.dropna().unique()
        values = [uniques[0], uniques[-1]] if len(uniques) else [None, None]

    # descrição (vamos supor que você já tenha uma descrição)
    description = 'Descrição da coluna'

    # adicionando informações à lista de dicionários
    col_info.append({
        'variavel': col,
        'descricao': description,
        'valores': values,
        'Tipo': col_type
    })

# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)

In [21]:
# Descrições de negócio, por nome da coluna em português
DESCRICOES = {
    "ordem": "Número de ordem do registro no arquivo original",
    "pid": "Identificador cadastral do lote na prefeitura (Parcel ID)",
    "subclasse": "Código do tipo de imóvel envolvido na venda (idade, pavimentos e estilo)",
    "zoneamento": "Classificação geral de zoneamento urbano do lote",
    "testada_lote": "Testada: pés lineares de rua conectados ao lote",
    "area_terreno": "Área do terreno, em pés quadrados",
    "tipo_via": "Tipo de pavimentação da via de acesso",
    "tipo_viela": "Tipo de pavimentação da viela de acesso (NA = sem viela)",
    "formato_lote": "Regularidade do formato do lote",
    "relevo": "Relevo do terreno (plano, encosta, depressão)",
    "infraestrutura": "Infraestrutura urbana disponível no lote",
    "config_lote": "Configuração do lote (interno, esquina, fundo de rua)",
    "declividade": "Declividade do terreno",
    "bairro": "Bairro de Ames em que o imóvel se localiza",
    "condicao_entorno_1": "Proximidade de via principal, ferrovia ou área de interesse",
    "condicao_entorno_2": "Segunda proximidade relevante, quando há mais de uma",
    "tipo_construcao": "Tipo de construção (isolada, geminada, sobreposta)",
    "estilo": "Estilo da residência e número de pavimentos",
    "qualidade_geral": "Qualidade geral do material e do acabamento (1 a 10)",
    "conservacao_geral": "Estado geral de conservação (1 a 10)",
    "ano_construcao": "Ano da construção original",
    "ano_reforma": "Ano da última reforma ou ampliação (igual à construção se nunca houve)",
    "estilo_telhado": "Estilo do telhado",
    "material_telhado": "Material do telhado",
    "revestimento_1": "Revestimento externo predominante",
    "revestimento_2": "Segundo revestimento externo, quando há mais de um material",
    "tipo_alvenaria_aparente": "Tipo de alvenaria aparente (None na fonte = Sem alvenaria aparente)",
    "area_alvenaria_aparente": "Área de alvenaria aparente, em pés quadrados",
    "qualidade_externa": "Qualidade do material de revestimento externo",
    "conservacao_externa": "Estado de conservação do revestimento externo",
    "fundacao": "Tipo de fundação",
    "qualidade_porao": "Altura e qualidade do porão (NA = sem porão)",
    "conservacao_porao": "Estado de conservação do porão (NA = sem porão)",
    "exposicao_porao": "Exposição do porão à luz externa ou saída para o jardim",
    "acabamento_porao_1": "Qualidade do acabamento da primeira área acabada do porão",
    "area_porao_acabada_1": "Área acabada tipo 1 do porão, em pés quadrados",
    "acabamento_porao_2": "Qualidade do acabamento da segunda área acabada do porão",
    "area_porao_acabada_2": "Área acabada tipo 2 do porão, em pés quadrados",
    "area_porao_bruta": "Área do porão sem acabamento, em pés quadrados",
    "area_porao": "Área total do porão, em pés quadrados (0 = sem porão)",
    "aquecimento": "Tipo de sistema de aquecimento",
    "qualidade_aquecimento": "Qualidade e condição do sistema de aquecimento",
    "ar_central": "Possui ar-condicionado central (Sim/Não)",
    "sistema_eletrico": "Tipo de sistema elétrico",
    "area_terreo": "Área do pavimento térreo, em pés quadrados",
    "area_pavimento_superior": "Área do pavimento superior, em pés quadrados (0 = pavimento único)",
    "area_acabamento_inferior": "Área acabada de baixa qualidade, somando todos os pavimentos",
    "area_habitavel": "Área habitável acima do solo, em pés quadrados",
    "banheiros_completos_porao": "Banheiros completos no porão",
    "lavabos_porao": "Lavabos no porão",
    "banheiros_completos": "Banheiros completos acima do solo",
    "lavabos": "Lavabos acima do solo",
    "dormitorios": "Dormitórios acima do solo (não inclui os do porão)",
    "cozinhas": "Cozinhas acima do solo",
    "qualidade_cozinha": "Qualidade da cozinha",
    "comodos": "Total de cômodos acima do solo, exceto banheiros",
    "funcionalidade": "Funcionalidade da residência, com dedução por limitações de uso",
    "lareiras": "Número de lareiras",
    "qualidade_lareira": "Qualidade da lareira (NA = sem lareira)",
    "tipo_garagem": "Posição da garagem em relação à casa",
    "ano_garagem": "Ano de construção da garagem",
    "acabamento_garagem": "Acabamento interno da garagem",
    "vagas_garagem": "Capacidade da garagem, em número de vagas",
    "area_garagem": "Área da garagem, em pés quadrados",
    "qualidade_garagem": "Qualidade da garagem (NA = sem garagem)",
    "conservacao_garagem": "Estado de conservação da garagem (NA = sem garagem)",
    "acesso_pavimentado": "Pavimentação do acesso de veículos",
    "area_deck": "Área de deck de madeira, em pés quadrados",
    "area_varanda_aberta": "Área de varanda aberta, em pés quadrados",
    "area_varanda_fechada": "Área de varanda fechada, em pés quadrados",
    "area_varanda_3estacoes": "Área de varanda de três estações, em pés quadrados",
    "area_varanda_tela": "Área de varanda com tela, em pés quadrados",
    "area_piscina": "Área da piscina, em pés quadrados (0 = sem piscina)",
    "qualidade_piscina": "Qualidade da piscina (NA = sem piscina)",
    "tipo_cerca": "Tipo e qualidade da cerca (NA = sem cerca)",
    "item_extra": "Item extra não coberto pelas demais colunas (galpão, quadra, elevador)",
    "valor_item_extra": "Valor do item extra, em dólares",
    "mes_venda": "Mês da venda (1 a 12)",
    "ano_venda": "Ano da venda (2006 a 2010)",
    "tipo_venda": "Modalidade da venda",
    "condicao_venda": "Condição da venda (normal, parcial, familiar, execução)",
    "preco_venda": "Preço de venda em dólares — rótulo reservado, fora da modelagem",
}
# Adicionando descrição para cada uma das variáveis selecionadas
descricoes = [DESCRICOES[original] for original in col_info_df["variavel"]]

assert len(descricoes) == len(col_info_df)

# Adicionando descrição para cada uma das variáveis selecionadas
for i in range(0, len(col_info_df)):
    col_info_df.loc[i, "descricao"] = descricoes[i]

col_info_df

,variavel,descricao,valores,Tipo
0,ordem,Número de ordem do registro no arquivo original,"[1, 2930]",Discreta
1,pid,Identificador cadastral do lote na prefeitura ...,"[526301100, 1007100110]",Discreta
2,subclasse,Código do tipo de imóvel envolvido na venda (i...,"[20, 190]",Discreta
3,zoneamento,Classificação geral de zoneamento urbano do lote,"[Residencial de baixa densidade, Agrícola]",Nominal
4,testada_lote,Testada: pés lineares de rua conectados ao lote,"[21.0, 313.0]",Contínua
...,...,...,...,...
77,mes_venda,Mês da venda (1 a 12),"[1, 12]",Temporal
78,ano_venda,Ano da venda (2006 a 2010),"[2006, 2010]",Temporal
79,tipo_venda,Modalidade da venda,"[Escritura convencional, Escritura com financi...",Nominal
80,condicao_venda,"Condição da venda (normal, parcial, familiar, ...","[Normal, Terreno adjacente]",Nominal


In [22]:
# Salvando dicionário de dados
col_info_df["valores"] = col_info_df["valores"].apply(
    lambda valores: [valor.item() if isinstance(valor, np.generic) else valor for valor in valores]
)
col_info_df.to_csv('../data/processed/dicionario.csv',index=False,sep=';')

In [23]:
# Salvando escalas ordinais
# Escala usada para as ordinais numéricas (marcadas como None no dicionário)
ESCALA_NUMERICA_PADRAO = list(range(1, 11))

# Convertendo o dicionário de escalas para o formato longo
linhas_escala = []
for coluna, niveis in ESCALAS_ORDINAIS.items():
    if niveis is None:
        niveis = ESCALA_NUMERICA_PADRAO
    for ordem, nivel in enumerate(niveis, start=1):
        linhas_escala.append({'coluna': coluna, 'nivel': nivel, 'ordem': ordem})

escala_df = pd.DataFrame(linhas_escala)

# Salvando o arquivo
escala_df.to_csv('../data/processed/escala.csv', index=False, encoding='utf-8', sep=';')